In [1]:
import os
from math import prod
from safetensors import safe_open
import plotly.graph_objects as go

# =====================================================================
# 1. Safetensors 텐서 카테고리별 용량 분석 함수
# =====================================================================
def analyze_tensor_categories_gib(file_path):
    if not os.path.exists(file_path): 
        return {"quantized_weights": 0.0, "quant_overhead": 0.0, "unquantized_layers": 0.0}
    
    categories = {"quantized_weights": 0.0, "quant_overhead": 0.0, "unquantized_layers": 0.0}
    
    with safe_open(file_path, framework="pt", device="cpu") as f:
        for key in f.keys():
            tensor = f.get_tensor(key)
            tensor_gib = (prod(tensor.shape) * tensor.element_size()) / (1024 ** 3)
            
            if "qweight" in key:
                categories["quantized_weights"] += tensor_gib
            elif any(x in key for x in ["scales", "qzeros", "g_idx"]):
                categories["quant_overhead"] += tensor_gib
            else:
                categories["unquantized_layers"] += tensor_gib
    return categories

# =====================================================================
# 2. 경로 설정 및 통합 데이터 모델링
# =====================================================================
BASE_ORIGINAL = r"C:\Users\user\SLM\00_Base_Models"
BASE_QUANT = r"C:\Users\user\SLM\02_cuda_aligned"

colors = {'Llama_3.2_1B': '#BDE0FE', 'Qwen2.5_1.5B': '#E1C6E9', 'TinyLlama_1.1B': '#FFE0B2'}
line_colors = {'Llama_3.2_1B': '#90CAF9', 'Qwen2.5_1.5B': '#CE93D8', 'TinyLlama_1.1B': '#FFB74D'}

models_setup = {
    "Llama_3.2_1B": [rf"{BASE_ORIGINAL}\Llama-3.2-1B-Instruct", "Distillation", "Llama 3.2"],
    "Qwen2.5_1.5B": [rf"{BASE_ORIGINAL}\Qwen2.5-1.5B-Instruct", "Base_RLHF", "Qwen 2.5"],
    "TinyLlama_1.1B": [rf"{BASE_ORIGINAL}\TinyLlama-1.1B-Chat-v1.0", "Base_Scratch", "TinyLlama"]
}

bit_labels = ["원본 (BF16)", "GPTQ 8-bit", "GPTQ 4-bit", "GPTQ 3-bit", "GPTQ 2-bit"]
bit_keys = ["Original", "GPTQ_8bit", "GPTQ_4bit", "GPTQ_3bit", "GPTQ_2bit"]

x_quant_level = [] 
x_model_name = []  

store = {"weights": [], "overhead": [], "unquantized": [], "colors": [], "lines": [], "weight_text": []}

base_weights = {}
for m_key, config in models_setup.items():
    res = analyze_tensor_categories_gib(rf"{config[0]}\model.safetensors")
    base_weights[m_key] = res["quantized_weights"] + res["unquantized_layers"]

for i, b_key in enumerate(bit_keys):
    quant_label = bit_labels[i]
    
    for m_key, config in models_setup.items():
        orig_path = config[0]
        sub_dir = config[1]
        short_name = config[2]
        
        path = rf"{orig_path}\model.safetensors" if i == 0 else rf"{BASE_QUANT}\{m_key}\{sub_dir}\{b_key}\model.safetensors"
        res = analyze_tensor_categories_gib(path)
        
        if i == 0:
            w_val = base_weights[m_key]
            o_val = 0.0
            u_val = 0.0
        else:
            w_val = res["quantized_weights"]
            o_val = res["quant_overhead"]
            u_val = res["unquantized_layers"]
            
        x_quant_level.append(quant_label)
        x_model_name.append(short_name)
        
        store["unquantized"].append(u_val)
        store["overhead"].append(o_val)
        store["weights"].append(w_val)
        store["colors"].append(colors[m_key])
        store["lines"].append(line_colors[m_key])
        
        if i == 0:
            store["weight_text"].append(f"<b>{w_val:.2f}</b>")
        else:
            reduction = ((base_weights[m_key] - w_val) / base_weights[m_key]) * 100
            store["weight_text"].append(f"<b>{w_val:.2f}</b><br><span style='font-size:10px;'>↓{reduction:.1f}%</span>")

multi_x_axis = [x_quant_level, x_model_name]

# =====================================================================
# 3. Plotly 통합 시각화 (중복 텍스트 제거)
# =====================================================================
fig = go.Figure()

# 레이어 1: 양자화 예외 레이어
fig.add_trace(go.Bar(
    name='양자화 예외 (Embedding, LM Head 등)', 
    x=multi_x_axis, y=store["unquantized"],
    marker_color='#E0E0E0', marker_line_color='#BDBDBD', marker_line_width=1,
    text=[f"{v:.2f}" if v > 0 else "" for v in store["unquantized"]],
    textposition='inside', textfont=dict(size=12, color='black'),
    showlegend=True, hovertemplate="%{x}<br>양자화 예외: %{y} GiB<extra></extra>" 
))

# 레이어 2: 양자화 오버헤드
fig.add_trace(go.Bar(
    name='양자화 오버헤드 (Metadata, Scales 등)', 
    x=multi_x_axis, y=store["overhead"],
    marker_color='#FFCDD2', marker_line_color='#EF9A9A', marker_line_width=1,
    text=[f"{v:.2f}" if v > 0.05 else "" for v in store["overhead"]],
    textposition='inside', textfont=dict(size=12, color='black'),
    showlegend=True, hovertemplate="%{x}<br>오버헤드: %{y} GiB<extra></extra>"
))

# 레이어 3: 순수 가중치
fig.add_trace(go.Bar(
    name='순수 모델 가중치', 
    x=multi_x_axis, y=store["weights"],
    marker_color=store["colors"], marker_line_color=store["lines"], marker_line_width=1.2,
    text=store["weight_text"], textposition='inside', textfont=dict(size=13, color='black'),
    showlegend=False, hovertemplate="%{x}<br>가중치 용량: %{y} GiB<extra></extra>"
))

# 총합(Total) 텍스트 오버레이
totals = [store["unquantized"][i] + store["overhead"][i] + store["weights"][i] for i in range(len(multi_x_axis[0]))]

# [수정됨] '원본' 데이터인 경우 바깥쪽(Top Center) 텍스트를 빈 문자열("")로 처리하여 중복 제거
total_labels = [
    "" if "원본" in multi_x_axis[0][i] else f"<b>{v:.2f}</b>" 
    for i, v in enumerate(totals)
]

fig.add_trace(go.Scatter(
    x=multi_x_axis, y=totals, mode='text',
    text=total_labels,
    textposition='top center', textfont=dict(size=14, color='#333333'),
    showlegend=False, hoverinfo='skip'
))

# 레이아웃 설정
fig.update_layout(
    title=dict(text='<b>양자화 단계별 디스크 용량 내부 구조 횡단 분석 (모델 대조)</b>', x=0.5, y=0.95, font=dict(size=22)),
    yaxis_title='<b>디스크 용량 (GiB)</b>', barmode='stack', height=850,
    font=dict(family="Malgun Gothic, AppleGothic, sans-serif"),
    plot_bgcolor='white',
    legend=dict(orientation="h", yanchor="top", y=-0.12, xanchor="center", x=0.5, font=dict(size=13), traceorder='reversed'),
    margin=dict(t=120, b=120, l=60, r=40)
)

fig.update_yaxes(showgrid=True, gridcolor='#F0F0F0', range=[0, max(totals) * 1.15])
fig.update_xaxes(
    tickfont=dict(size=12, weight='bold'), 
    tickangle=0, 
    dividercolor='#EAEAEA', dividerwidth=2 
)

fig.show()

In [2]:
import os
from math import prod
from safetensors import safe_open
import plotly.graph_objects as go

# =====================================================================
# 1. 데이터 추출 함수 (순수 텐서 용량 기준)
# =====================================================================
def analyze_safetensors_gib(file_path):
    if not os.path.exists(file_path): return 0.0
    tensor_bytes_sum = 0
    with safe_open(file_path, framework="pt", device="cpu") as f:
        for key in f.keys():
            tensor = f.get_tensor(key)
            params = prod(tensor.shape)
            element_size = tensor.element_size() 
            tensor_bytes_sum += (params * element_size)
    return tensor_bytes_sum / (1024 ** 3)

# =====================================================================
# 2. 경로 설정 및 데이터 처리
# =====================================================================
BASE_ORIGINAL = r"C:\Users\user\SLM\00_Base_Models"
BASE_QUANT = r"C:\Users\user\SLM\02_cuda_aligned"

models_config = {
    "Llama_3.2_1B": [
        rf"{BASE_ORIGINAL}\Llama-3.2-1B-Instruct\model.safetensors",
        rf"{BASE_QUANT}\Llama_3.2_1B\Distillation\GPTQ_8bit\model.safetensors",
        rf"{BASE_QUANT}\Llama_3.2_1B\Distillation\GPTQ_4bit\model.safetensors",
        rf"{BASE_QUANT}\Llama_3.2_1B\Distillation\GPTQ_3bit\model.safetensors",
        rf"{BASE_QUANT}\Llama_3.2_1B\Distillation\GPTQ_2bit\model.safetensors"
    ],
    "Qwen2.5_1.5B": [
        rf"{BASE_ORIGINAL}\Qwen2.5-1.5B-Instruct\model.safetensors",
        rf"{BASE_QUANT}\Qwen2.5_1.5B\Base_RLHF\GPTQ_8bit\model.safetensors",
        rf"{BASE_QUANT}\Qwen2.5_1.5B\Base_RLHF\GPTQ_4bit\model.safetensors",
        rf"{BASE_QUANT}\Qwen2.5_1.5B\Base_RLHF\GPTQ_3bit\model.safetensors",
        rf"{BASE_QUANT}\Qwen2.5_1.5B\Base_RLHF\GPTQ_2bit\model.safetensors"
    ],
    "TinyLlama_1.1B": [
        rf"{BASE_ORIGINAL}\TinyLlama-1.1B-Chat-v1.0\model.safetensors",
        rf"{BASE_QUANT}\TinyLlama_1.1B\Base_Scratch\GPTQ_8bit\model.safetensors",
        rf"{BASE_QUANT}\TinyLlama_1.1B\Base_Scratch\GPTQ_4bit\model.safetensors",
        rf"{BASE_QUANT}\TinyLlama_1.1B\Base_Scratch\GPTQ_3bit\model.safetensors",
        rf"{BASE_QUANT}\TinyLlama_1.1B\Base_Scratch\GPTQ_2bit\model.safetensors"
    ]
}

reduction_rates = {}
for model, paths in models_config.items():
    sizes = [analyze_safetensors_gib(p) for p in paths]
    base_size = sizes[0]
    
    if base_size > 0:
        rates = [((base_size - s) / base_size) * 100 if s > 0 else 0 for s in sizes[1:]]
    else:
        rates = [0.0, 0.0, 0.0, 0.0]
        
    reduction_rates[model] = rates

# =====================================================================
# 3. Plotly 그룹형 막대 + 이론적 상한선 오버레이
# =====================================================================
labels_quant_only = ['GPTQ 8-bit', 'GPTQ 4-bit', 'GPTQ 3-bit', 'GPTQ 2-bit']
theoretical_rates = [50.0, 75.0, 81.25, 87.5]

# 지정된 막대 채우기 및 테두리 색상
colors = {'Llama_3.2_1B': '#BDE0FE', 'Qwen2.5_1.5B': '#E1C6E9', 'TinyLlama_1.1B': '#FFE0B2'}
line_colors = {'Llama_3.2_1B': '#5A9BD5', 'Qwen2.5_1.5B': '#9B59B6', 'TinyLlama_1.1B': '#E67E22'}

fig = go.Figure()

# 1. 실제 모델들의 감소율 막대그래프
for model_name, rates in reduction_rates.items():
    text_labels = [f"<b>{rate:.1f}%</b>" if rate > 0 else "" for rate in rates]
    
    fig.add_trace(go.Bar(
        name=model_name,
        x=labels_quant_only,
        y=rates,
        text=text_labels,
        textposition='inside', 
        insidetextanchor='end',
        textfont=dict(size=14, color='#000000'), 
        marker_color=colors[model_name],
        marker_line_color=line_colors[model_name], 
        marker_line_width=1.5,
        hovertemplate="<b>%{x}</b><br>모델: " + model_name + "<br>실제 감소율: %{y:.2f}%<extra></extra>"
    ))

# 2. 이론적 상한선(Ceiling) 점선 그래프 오버레이
fig.add_trace(go.Scatter(
    name='이론적 최대 압축률 (수학적 한계선)',
    x=labels_quant_only,
    y=theoretical_rates,
    mode='lines+markers+text',
    text=[f"<b>이론치: {rate:.1f}%</b>" for rate in theoretical_rates],
    textposition='top center',
    textfont=dict(size=13, color='#D32F2F'),
    line=dict(color='#D32F2F', width=3, dash='dot'), 
    marker=dict(symbol='diamond', size=10, color='white', line=dict(color='#D32F2F', width=2)),
    hovertemplate="<b>%{x}</b><br>이론적 최대 감소율: %{y:.2f}%<extra></extra>"
))

# # =====================================================================
# # 4. [핵심 추가] '오버헤드 격차(Gap)' 시각적 주석(Annotation) 추가
# # =====================================================================
# # 가장 오버헤드가 큰 Llama 3.2 1B의 2-bit 구간에 화살표와 텍스트 배치
# target_actual_rate = reduction_rates['Llama_3.2_1B'][3] if 'Llama_3.2_1B' in reduction_rates else 68.2
# target_theo_rate = theoretical_rates[3] # 87.5

# fig.add_annotation(
#     x='GPTQ 2-bit',              # X축 기준점
#     y=target_theo_rate - 1,      # 화살표 상단 끝점 (이론선 바로 아래)
#     ax='GPTQ 2-bit',
#     ay=target_actual_rate + 1,   # 화살표 하단 끝점 (막대 바로 위)
#     xref='x', yref='y', axref='x', ayref='y',
#     text="<b>이론과 실제의 격차<br>= 고정 오버헤드</b>", # 설명 캡션
#     font=dict(size=12, color='#D32F2F'),
#     showarrow=True,
#     arrowhead=2,
#     arrowside='end+start',       # ★ 양방향 화살표(↕) 설정
#     arrowsize=1.5,
#     arrowwidth=2,
#     arrowcolor='#D32F2F',
#     xshift=-55,                  # Llama가 3개 그룹 중 가장 왼쪽 막대이므로 중심축 이동
#     align='center',
#     bgcolor='rgba(255, 255, 255, 0.8)', # 격자 선에 묻히지 않도록 반투명 배경 추가
#     bordercolor='#D32F2F',
#     borderwidth=1,
#     borderpad=3
# )

# =====================================================================
# 5. 레이아웃 최적화 (범례 하단 배치)
# =====================================================================
fig.update_layout(
    title=dict(text='<b>모델별 양자화 압축 효율(%) 비교 및 이론적 한계선(Ceiling) 분석</b>', font=dict(size=22), x=0.5, y=0.95),
    yaxis_title=dict(text='<b>용량 감소율 (%)</b>', font=dict(size=15)),
    barmode='group',
    bargroupgap=0.15,
    bargap=0.2,
    yaxis=dict(
        range=[0, 105], 
        showgrid=True, gridcolor='#EAEAEA', zeroline=True, zerolinecolor='#CCCCCC'
    ),
    xaxis=dict(tickfont=dict(size=14, weight='bold')),
    font=dict(family="Malgun Gothic, AppleGothic, NanumGothic, sans-serif"),
    plot_bgcolor='white',
    
    # [수정됨] 범례를 X축 아래(-0.15) 중앙(0.5)으로 이동
    legend=dict(
        orientation="h", 
        yanchor="top", 
        y=-0.15, 
        xanchor="center", 
        x=0.5, 
        font=dict(size=13)
    ),
    
    # [수정됨] 상단 여백(t)을 줄이고 하단 여백(b)을 늘려 범례 공간 확보
    margin=dict(t=80, b=100, l=50, r=50)
)

fig.show()